# HCMC flood observations — Method 2 / Random Forest pilot

Adaptation of the supplied **Phase 3 RF notebook** (`notebook50b59dca3c.ipynb`). Retains RandomForestRegressor with 10 trees and seed 42; replaces raster sampling and 13 Berlin predictors with a reader and explicit feature selection for the original HCMC tabular file. Trains a new RF; Berlin weights are not transferred. The original data file remains unchanged.

**Target:** the reported upper depth endpoint, `Water height max (cm)` / 100. This is NOT necessarily measured peak flood depth. Missing upper endpoints are excluded, never filled using the lower endpoint or a class. This pilot describes severity among reported flood observations, not flood occurrence or a physical rainfall forecast.

Source: Hardy & Radziute (2026), https://doi.org/10.23708/8Y16HU, V1, CC BY-NC 4.0. Dataset contains historical newspaper/report observations. Code reference: https://github.com/omarseleem92/Machine_learning_for_urban_flooding ; paper https://doi.org/10.5194/nhess-23-809-2023 .

Kaggle: CPU is sufficient. Put `Flood_event_locations.tab` beside this notebook or attach it as an input. If absent, the notebook downloads the public original file (Internet required). Run all cells. Dependencies: numpy, pandas, scikit-learn.


In [1]:
from pathlib import Path
import json, hashlib, urllib.request, unicodedata, re, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = BASE / 'hcm_method2_results'
OUT.mkdir(exist_ok=True)
DATA = BASE / 'Flood_event_locations.tab'
if not DATA.exists() and Path('/kaggle/input').exists():
    candidates = sorted(Path('/kaggle/input').rglob('Flood_event_locations.tab'))
    if candidates: DATA = candidates[0]
if not DATA.exists():
    data_bytes = urllib.request.urlopen('https://dataverse.ird.fr/api/access/datafile/50148', timeout=60).read()
    DATA.write_bytes(data_bytes)
SOURCE_HASH = hashlib.sha256(DATA.read_bytes()).hexdigest()
raw = pd.read_csv(DATA, sep='\t', dtype=str, keep_default_na=False)
print('Original dataset:', raw.shape, '| SHA256:', SOURCE_HASH)
print('Versions:', platform.python_version(), pd.__version__, sklearn.__version__)


Original dataset: (425, 22) | SHA256: 4c68cab32de88accc6d582e42d62de1c9b19483e9527af9cf883c76a36098cd0
Versions: 3.11.0 3.1.1 1.9.1


## Audit and target
Rows without a usable upper endpoint are excluded. A repeated Obs_id is reported but is not automatically considered a duplicated record. Exact duplicate substantive rows are removed only in memory. No data-dependent model selection uses the evaluation folds.


In [2]:
target_col = 'Water height max (cm)'
upper = pd.to_numeric(raw[target_col], errors='coerce')
lower = pd.to_numeric(raw['Water height min (cm)'], errors='coerce')
valid = upper.notna() & (upper >= 0) & (~lower.notna() | (lower <= upper))
df = raw.loc[valid].copy()
substantive = [c for c in raw.columns if c not in ['OBJECTID', 'Obs_id']]
n_before = len(df)
df = df.drop_duplicates(subset=substantive).reset_index(drop=True)
y = pd.to_numeric(df[target_col]).to_numpy(dtype=float) / 100.0
audit = {'original_rows':len(raw), 'usable_upper_endpoint_rows':int(valid.sum()),
         'excluded_rows':int((~valid).sum()), 'exact_duplicate_rows_removed':n_before-len(df),
         'duplicate_obs_id_rows':int(raw['Obs_id'].duplicated().sum()),
         'target_min_m':float(y.min()), 'target_max_m':float(y.max()),
         'cause_counts':df['Cause'].value_counts().to_dict(), 'sha256':SOURCE_HASH}
print(json.dumps(audit, ensure_ascii=False, indent=2))
(OUT/'data_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')


{
  "original_rows": 425,
  "usable_upper_endpoint_rows": 147,
  "excluded_rows": 278,
  "exact_duplicate_rows_removed": 0,
  "duplicate_obs_id_rows": 1,
  "target_min_m": 0.1,
  "target_max_m": 3.0,
  "cause_counts": {
    "High tide": 76,
    "Rain": 54,
    "Combined": 17
  },
  "sha256": "4c68cab32de88accc6d582e42d62de1c9b19483e9527af9cf883c76a36098cd0"
}


361

## Features and grouped evaluation
This remains a context-only severity pilot: year, month, location type, district and province, with an optional retrospective reported-cause variant. There is no rainfall, elevation or river-distance predictor yet, so these scores do not test H1.

The primary split uses connected components: observations sharing a nonempty named location, source ID, or parsed date are assigned to the same group, including transitive links. The date is a conservative event proxy because verified event IDs are unavailable. Missing names and IDs do not link unrelated rows. All models use the same GroupKFold folds.


In [3]:
def norm(v):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(v)).strip().lower())

dates = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
X = pd.DataFrame({'Year':pd.to_numeric(df['Year'],errors='coerce').fillna(0),
                  'Month':dates.dt.month.fillna(0)})
cats = ['Location type','District mentioned','Province']
for c in cats+['Cause']: X[c] = df[c].map(lambda v: norm(v) or '__missing__')

parent = list(range(len(df)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

seen = {}
for i in range(len(df)):
    keys = []
    location = norm(df.loc[i, 'Location name'])
    source = norm(df.loc[i, 'Source id'])
    if location: keys.append('location:' + location)
    if source: keys.append('source:' + source)
    if pd.notna(dates.iloc[i]): keys.append('date:' + dates.iloc[i].strftime('%Y-%m-%d'))
    for key in keys:
        if key in seen: parent[find(i)] = find(seen[key])
        else: seen[key] = i
groups = np.array([find(i) for i in range(len(df))])
print('Merged location/source/date components:', len(set(groups)))
print('Largest component:', int(pd.Series(groups).value_counts().max()))
print('Unparsed or missing dates:', int(dates.isna().sum()))


Merged location/source/date components: 10
Largest component: 117
Unparsed or missing dates: 24


In [4]:
scores=[]
predictions=[]
fold_scores=[]
for protocol,groups in [('merged_location_event_source',groups)]:
    n_splits=min(5,len(set(groups)))
    if n_splits < 3: raise ValueError('Too few independent groups')
    splits=list(GroupKFold(n_splits=n_splits).split(X,y,groups))
    for model_name in ['mean_baseline','rf_context','rf_context_and_known_cause']:
        categorical=cats+(['Cause'] if model_name.endswith('known_cause') else [])
        columns=['Year','Month']+categorical
        oof=np.full(len(y),np.nan)
        for fold,(train,test) in enumerate(splits):
            assert not set(groups[train]) & set(groups[test])
            if model_name=='mean_baseline':
                model=DummyRegressor(strategy='mean')
            else:
                prep=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),categorical),
                                        ('num','passthrough',['Year','Month'])])
                model=Pipeline([('prepare',prep),('rf',RandomForestRegressor(
                    n_estimators=10,random_state=42,n_jobs=4))])
            model.fit(X.iloc[train][columns], y[train])
            pred=model.predict(X.iloc[test][columns])
            assert np.isfinite(pred).all()
            oof[test]=pred
            fold_scores.append(dict(protocol=protocol,model=model_name,fold=fold,n_train=len(train),
                                    n_test=len(test),mae_m=mean_absolute_error(y[test],pred),
                                    rmse_m=float(np.sqrt(mean_squared_error(y[test],pred)))))
            for i,p in zip(test,pred):
                predictions.append(dict(protocol=protocol,model=model_name,fold=fold,
                    object_id=df.loc[i,'OBJECTID'],obs_id=df.loc[i,'Obs_id'],
                    actual_upper_m=y[i],predicted_upper_m=float(p)))
        assert np.isfinite(oof).all()
        scores.append(dict(protocol=protocol,model=model_name,n=len(y),n_folds=n_splits,
            mae_m=mean_absolute_error(y,oof),rmse_m=float(np.sqrt(mean_squared_error(y,oof))),
            r2=r2_score(y,oof)))
metrics=pd.DataFrame(scores)
print(metrics.to_string(index=False))
metrics.to_csv(OUT/'metrics.csv',index=False)
pd.DataFrame(fold_scores).to_csv(OUT/'fold_metrics.csv',index=False)
pd.DataFrame(predictions).to_csv(OUT/'out_of_fold_predictions.csv',index=False)
assert hashlib.sha256(DATA.read_bytes()).hexdigest()==SOURCE_HASH


                    protocol                      model   n  n_folds    mae_m   rmse_m        r2
merged_location_event_source              mean_baseline 147        5 0.426634 0.579174 -0.521994
merged_location_event_source                 rf_context 147        5 0.454588 0.630623 -0.804408
merged_location_event_source rf_context_and_known_cause 147        5 0.467825 0.628442 -0.791948


## Interpretation
Compare each RF with the mean baseline on identical merged-group folds. This split prevents observed location, source and parsed-date links crossing train and test. The date proxy can merge separate events on one day, and unidentified links may remain. The reported upper depth endpoint is not necessarily measured peak depth.

This pilot has no rainfall, elevation, or river-distance inputs and cannot test the proposed rainfall-only versus rainfall plus historical and spatial hypothesis. Depth-class classification is a separate follow-up for rows without a numeric upper endpoint.


In [5]:
summary={'status':'EXECUTED_HCMC_RF_METHOD2_PILOT',
 'source_notebook':'notebook50b59dca3c.ipynb (Berlin Phase 3 RF)',
 'adaptation':'new RF trained on HCMC tabular observations; no pretrained Berlin weights',
 'target':'reported upper depth endpoint in metres; not necessarily observed peak',
 'features_main':['Year','Month']+cats, 'extra_retrospective_feature':'Cause',
 'grouping':'transitive nonempty location name OR source ID OR parsed date', 'n_groups':len(set(groups)), 'audit':audit, 'metrics':metrics.to_dict(orient='records'),
 'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__}}
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Finished. Outputs:',OUT)


Finished. Outputs: C:\Users\ADMIN\Documents\ChatGPT\SC203\flood-repo\hcm\hcm_method2_results


## Observed result

The executed metrics are in the table above and `results/metrics.csv`. Report only the merged-group scores as the current pilot evaluation. The previous location-only R² of 0.7059 is invalid as an independent-event claim because event/source links could cross folds. These context-only models do not test H1.
